# Split `attom_vt.parquet` into DuckDB chunks, then convert to Stata `.dta`

This notebook keeps **all variables** from `data/attom_vt.parquet`, but writes them in manageable chunks so the conversion can be monitored and resumed.

Outputs:
- Chunk parquet files: `data/build/vt_attom_full_chunks/parquet/`
- Chunk Stata files: `data/build/vt_attom_full_chunks/dta/`


In [ ]:
from pathlib import Path
import math
import re

import duckdb
import numpy as np
import pandas as pd
import pyreadstat

ROOT = Path('/Users/anna/Desktop/climate-investments')
INPUT = ROOT / 'data' / 'attom_vt.parquet'
OUT_ROOT = ROOT / 'data' / 'build' / 'vt_attom_full_chunks'
PARQUET_DIR = OUT_ROOT / 'parquet'
DTA_DIR = OUT_ROOT / 'dta'

# 500k rows produced ~1GB .dta chunks in testing. Lower this if memory feels tight.
CHUNK_ROWS = 500_000

PARQUET_DIR.mkdir(parents=True, exist_ok=True)
DTA_DIR.mkdir(parents=True, exist_ok=True)

INPUT, OUT_ROOT

In [ ]:
def q(value: str) -> str:
    return "'" + str(value).replace("'", "''") + "'"

def stata_safe_names(columns):
    out = {}
    used = set()
    for col in columns:
        base = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        if not re.match(r'[A-Za-z_]', base):
            base = 'v_' + base
        base = base[:32]
        name = base
        i = 1
        while name.lower() in used:
            suffix = f'_{i}'
            name = base[: 32 - len(suffix)] + suffix
            i += 1
        used.add(name.lower())
        out[col] = name
    return out

def clean_for_stata(df):
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].fillna('').astype(str)
    for col in df.select_dtypes(include=['bool']).columns:
        df[col] = df[col].astype(np.int8)
    return df.rename(columns=stata_safe_names(list(df.columns)))

## 1. Inspect the parquet

In [ ]:
con = duckdb.connect()
n_rows = con.execute(f"SELECT count(*) FROM read_parquet({q(INPUT)})").fetchone()[0]
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet({q(INPUT)}) LIMIT 1").df()
n_parts = math.ceil(n_rows / CHUNK_ROWS)
print(f'Input: {INPUT}')
print(f'Rows: {n_rows:,}')
print(f'Columns: {len(schema):,}')
print(f'Chunk rows: {CHUNK_ROWS:,}')
print(f'Parts: {n_parts:,}')
schema.head(20)

## 2. Split parquet with DuckDB

This writes one parquet file per chunk. It skips chunks already written, so this cell is safe to rerun.

In [ ]:
for part in range(1, n_parts + 1):
    offset = (part - 1) * CHUNK_ROWS
    limit = min(CHUNK_ROWS, n_rows - offset)
    out = PARQUET_DIR / f'vt_attom_full_part{part:03d}.parquet'
    if out.exists() and out.stat().st_size > 0:
        print(f'Skip parquet part {part:03d}/{n_parts:03d}: exists ({out.stat().st_size/1024**2:.1f} MiB)')
        continue
    print(f'Write parquet part {part:03d}/{n_parts:03d}: rows {offset + 1:,}-{offset + limit:,}')
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_parquet({q(INPUT)})
            LIMIT {limit} OFFSET {offset}
        ) TO {q(out)} (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
print('Parquet split complete.')

## 3. Convert parquet chunks to `.dta`

This preserves all variables in each chunk. It also skips `.dta` chunks already written.

In [ ]:
for part in range(1, n_parts + 1):
    src = PARQUET_DIR / f'vt_attom_full_part{part:03d}.parquet'
    out = DTA_DIR / f'vt_attom_full_part{part:03d}.dta'
    if out.exists() and out.stat().st_size > 0:
        print(f'Skip dta part {part:03d}/{n_parts:03d}: exists ({out.stat().st_size/1024**2:.1f} MiB)')
        continue
    print(f'Read parquet part {part:03d}/{n_parts:03d}: {src.name}')
    df = pd.read_parquet(src)
    print(f'  Loaded {len(df):,} rows x {len(df.columns):,} columns')
    df = clean_for_stata(df)
    print(f'  Write {out.name}')
    pyreadstat.write_dta(df, str(out), version=15)
    del df
print('DTA conversion complete.')

## 4. Verify outputs

In [ ]:
dta_files = sorted(DTA_DIR.glob('vt_attom_full_part*.dta'))
print(f'DTA files: {len(dta_files):,}')
for p in dta_files:
    print(f'{p.name:32s} {p.stat().st_size / 1024**2:,.1f} MiB')

if dta_files:
    _, meta = pyreadstat.read_dta(str(dta_files[0]), metadataonly=True)
    print('\nFirst part metadata:')
    print(f'Rows: {meta.number_rows:,}')
    print(f'Columns: {meta.number_columns:,}')

## Optional Stata append pattern

If you later want to append the chunk files in Stata:

```stata
clear
local files : dir "/Users/anna/Desktop/climate-investments/data/build/vt_attom_full_chunks/dta" files "vt_attom_full_part*.dta"
local first = 1
foreach f of local files {
    if `first' == 1 {
        use "/Users/anna/Desktop/climate-investments/data/build/vt_attom_full_chunks/dta/`f'", clear
        local first = 0
    }
    else {
        append using "/Users/anna/Desktop/climate-investments/data/build/vt_attom_full_chunks/dta/`f'"
    }
}
```
